In [28]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

contrast_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)


class ClfDS(Dataset):
    def __init__(self, folder):
        self.folder = folder
        classes=[]
        images=[]
        for cls in os.listdir(self.folder):
          for files in os.listdir(os.path.join(self.folder, cls)):
            img_path =os.path.join(self.folder, cls, files)
            images.append(img_path)
            classes.append(cls)
        self.id2label = {idx: label for idx, label in enumerate(sorted(set(classes)))}
        self.label2id = {label: idx for idx, label in enumerate(sorted(set(classes)))}
        self.classes = classes
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        id=self.classes[idx]
        img = Image.open(img_path)
        img = contrast_transforms(img)

        return img, self.label2id[id]

In [29]:
from pathlib import Path

base = Path(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7')
train_cls = {p.name for p in (base / 'train').iterdir() if p.is_dir()}
val_cls   = {p.name for p in (base / 'valid').iterdir() if p.is_dir()}

print("train:", len(train_cls), "val:", len(val_cls))
print("faqat trainda:", sorted(train_cls - val_cls))
print("faqat valda:  ", sorted(val_cls - train_cls))

train: 172 val: 172
faqat trainda: []
faqat valda:   []


In [32]:
from torch.utils.data import DataLoader
import os
train_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\train'), batch_size=32, shuffle=True)
val_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\valid'), batch_size=32, shuffle=False)

In [31]:
next(iter(val_dl))

[tensor([[[[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [0.9451, 0.9529, 0.9647,  ..., 1.0000, 0.9961, 1.0000],
           [0.8980, 0.9020, 0.9098,  ..., 1.0000, 0.9961, 0.9961],
           ...,
           [0.9804, 0.9765, 0.9804,  ..., 1.0000, 1.0000, 1.0000],
           [0.8667, 0.8627, 0.8745,  ..., 1.0000, 1.0000, 1.0000],
           [0.6784, 0.6706, 0.6706,  ..., 1.0000, 1.0000, 1.0000]],
 
          [[0.6196, 0.6157, 0.6000,  ..., 0.3529, 0.3490, 0.3569],
           [0.4118, 0.4196, 0.4275,  ..., 0.3529, 0.3490, 0.3529],
           [0.3333, 0.3373, 0.3412,  ..., 0.3569, 0.3490, 0.3529],
           ...,
           [0.9569, 0.9490, 0.9373,  ..., 0.3137, 0.3137, 0.3137],
           [0.7804, 0.7765, 0.7843,  ..., 0.3137, 0.3137, 0.3137],
           [0.5686, 0.5608, 0.5647,  ..., 0.3137, 0.3137, 0.3137]],
 
          [[0.2941, 0.2902, 0.2784,  ..., 0.0078, 0.0078, 0.0196],
           [0.0902, 0.0980, 0.1059,  ..., 0.0078, 0.0000, 0.0118],
           [0.0157, 0.01

In [ ]:
import timm
import torch

vit_model = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=len(set(val_dl.dataset.classes)))

In [39]:
len(set(val_dl.dataset.classes))

172

In [34]:
import pytorch_lightning as L


class LitAutoEncoder(L.LightningModule):
    def __init__(
            self,
    ):
        super().__init__()
        self.model = vit_model

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        images, labels = batch
        preds = self.model(images)

        loss = F.cross_entropy(preds, labels)

        self.log("Loss", loss.item(), prog_bar=True)

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer


vit_model = LitAutoEncoder()

In [35]:
trainer = L.Trainer(max_epochs=10, check_val_every_n_epoch=1)
trainer.fit(model=vit_model, train_dataloaders=train_dl, val_dataloaders=val_dl)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ VisionTransformer │  5.6 M │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 5.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.6 M                                                                                                
Total estimated model params size (MB): 22.230                                                                     
Modules in train mode: 276                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


`Trainer.fit` stopped: `max_epochs=10` reached.


In [36]:
torch.save(vit_model.model, "CE_VIT_clf.pth")

In [37]:
model=torch.load(r"C:\Users\Administrator\Desktop\Coding\CE_VIT_clf.pth", weights_only=False)
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)


In [38]:
clf_model = model.cuda()
import torch
import torch.nn.functional as F

from sklearn.metrics import classification_report
import numpy as np

all_preds = []
all_labels = []

for batch in val_dl:
    images, labels = batch
    val_out = clf_model(images.cuda())

    probs = F.softmax(val_out, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()
    labels = labels.cpu().numpy()

    preds = [val_dl.dataset.id2label[idx] for idx in preds]
    labels = [val_dl.dataset.id2label[idx] for idx in labels]

    all_preds.extend(preds)
    all_labels.extend(labels)

print(classification_report(all_labels, all_preds))

                          precision    recall  f1-score   support

                n_1_1050       1.00      0.06      0.11        17
              n_1_1050-2       0.43      0.85      0.57        27
              n_1_1050-3       0.93      0.79      0.86       139
                 n_1_300       0.83      0.78      0.81        79
               n_1_300-2       0.60      0.79      0.68        71
               n_1_300-3       0.91      0.65      0.76       182
                 n_1_600       0.91      0.78      0.84        40
               n_1_600-2       0.61      0.75      0.67        48
               n_1_600-3       0.73      0.84      0.78       163
                n_2_1050       1.00      0.44      0.62         9
              n_2_1050-2       0.62      0.56      0.59        32
              n_2_1050-3       0.88      0.75      0.81       106
                 n_2_300       0.72      0.84      0.77        81
               n_2_300-2       0.60      0.54      0.57        65
         

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital